In [ ]:
# Prescription Medicine Classification with PyTorch
# =============================================

# 1. Setup and Imports
# ------------------
import pandas as pd
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [4]:
# 2. Check for GPU
# --------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

Using device: cpu
PyTorch version: 2.6.0+cu124
GPU Available: False


In [5]:
# 3. Configuration
# --------------
class Config:
    BATCH_SIZE = 16  # Reduced batch size to improve generalization
    EPOCHS = 40
    EARLY_STOPPING_PATIENCE = 5  # Stop training if validation doesn't improve
    IMG_SIZE = 224
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    LR = 5e-4  # Reduced learning rate to prevent quick overfitting
    WEIGHT_DECAY = 1e-3  # Increased weight decay for more regularization
    MODEL_SAVE_PATH = 'best_model.pth'
    DROPOUT_RATE = 0.5  # Increased dropout for regularization

cfg = Config()

In [6]:
# 4. Custom Dataset Class
# ---------------------
class PrescriptionDataset(Dataset):
    def __init__(self, data_dir, label_path, img_size=224, is_training=False, class_to_idx=None):
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_training = is_training

        # Check if data directory exists
        if not os.path.exists(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        # Check if label file exists
        if not os.path.exists(label_path):
            raise FileNotFoundError(f"Label file not found: {label_path}")

        # Load and validate data
        try:
            # Try first with header
            self.df = pd.read_csv(label_path)
            # Check if columns exist, if not try reading without header
            if 'IMAGE' not in self.df.columns:
                self.df = pd.read_csv(label_path, header=None,
                                    names=['IMAGE', 'MEDICINE_NAME', 'GENERIC_NAME'])
        except Exception as e:
            print(f"Error in initial CSV loading, trying alternative format: {str(e)}")
            try:
                # Alternative: skip first row and assign column names
                self.df = pd.read_csv(label_path, skiprows=1,
                                    names=['IMAGE', 'MEDICINE_NAME', 'GENERIC_NAME'])
            except Exception as e2:
                raise ValueError(f"Failed to load CSV: {str(e2)}")

        # Clean up data
        self.df = self.df.dropna(subset=['IMAGE', 'MEDICINE_NAME'])
        self.df = self.df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

        # File extension handling
        self.df['IMAGE'] = self.df['IMAGE'].apply(
            lambda x: f"{x}.png" if not str(x).lower().endswith(('.png', '.jpg', '.jpeg')) else x
        )

        # Validate images
        self.valid_samples = []
        missing_files = []
        for idx, row in self.df.iterrows():
            img_path = os.path.join(data_dir, row['IMAGE'])
            if os.path.exists(img_path):
                self.valid_samples.append(row)
            else:
                missing_files.append(img_path)

        if missing_files:
            print(f"Warning: {len(missing_files)} missing image files")
            if len(missing_files) < 5:  # Show a few examples
                print(f"Examples: {missing_files[:3]}")

        self.df = pd.DataFrame(self.valid_samples)
        if len(self.df) == 0:
            raise ValueError(f"No valid images found in {data_dir}")

        # Class handling
        if class_to_idx:
            self.class_to_idx = class_to_idx
            self.classes = list(class_to_idx.keys())
            self.idx_to_class = {v: k for k, v in class_to_idx.items()}
            # Keep only samples with labels in provided classes
            self.df = self.df[self.df['MEDICINE_NAME'].isin(self.classes)]
            if len(self.df) == 0:
                raise ValueError(f"No valid samples found with provided class labels in {data_dir}")
        else:
            self.classes = sorted(self.df['MEDICINE_NAME'].unique())
            self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
            self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}

        print(f"\nLoaded {len(self.df)} samples from {data_dir}")
        print(f"Classes: {len(self.classes)}")
        print("Class distribution:")
        class_dist = self.df['MEDICINE_NAME'].value_counts()
        print(class_dist.head())

        # Check for class imbalance
        if len(class_dist) > 1:
            min_samples = class_dist.min()
            max_samples = class_dist.max()
            if max_samples > 5 * min_samples:
                print(f"Warning: Significant class imbalance detected. Min: {min_samples}, Max: {max_samples}")

        # Define transformations
        if is_training:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(p=0.3),
                transforms.RandomRotation(degrees=30),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
                transforms.GaussianBlur(5, sigma=(0.1, 2.0)),
                transforms.ToTensor(),
                transforms.Normalize(mean=cfg.MEAN, std=cfg.STD)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                transforms.Normalize(mean=cfg.MEAN, std=cfg.STD)
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.data_dir, row['IMAGE'])

        try:
            # Read image
            img = cv2.imread(img_path)
            if img is None:
                raise FileNotFoundError(f"Could not load {img_path}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size))

            # Apply transformations
            img_tensor = self.transform(img)

            # Get label
            label = self.class_to_idx[row['MEDICINE_NAME']]

            return img_tensor, label

        except Exception as e:
            print(f"Error processing image {img_path}: {str(e)}")
            # Return zeros for image if error occurs
            return torch.zeros((3, self.img_size, self.img_size)), self.class_to_idx[row['MEDICINE_NAME']]

In [7]:
# 5. Create Model Architecture
# --------------------------
class PrescriptionModel(nn.Module):
    def __init__(self, num_classes):
        super(PrescriptionModel, self).__init__()

        # Load pre-trained EfficientNetV2S
        weights = EfficientNet_V2_S_Weights.DEFAULT
        self.base_model = efficientnet_v2_s(weights=weights)

        # Get number of features from last layer
        num_features = self.base_model.classifier[1].in_features

        # Replace classifier
        self.base_model.classifier = nn.Identity()

        # Create new classifier
        self.classifier = nn.Sequential(
            nn.Dropout(cfg.DROPOUT_RATE),
            nn.Linear(num_features, 640),
            nn.ReLU(),
            nn.BatchNorm1d(640),
            nn.Dropout(cfg.DROPOUT_RATE),
            nn.Linear(640, num_classes)
        )

        # Freeze early layers
        for i, param in enumerate(self.base_model.parameters()):
            if i < 100:  # Freeze first 100 layers
                param.requires_grad = False

    def forward(self, x):
        features = self.base_model(x)
        output = self.classifier(features)
        return output

In [8]:
# 6. Training Function
# ------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=25):
    best_val_acc = 0.0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    # Add learning rate to history if scheduler is provided
    if scheduler:
        history['lr'] = []

    # Early stopping parameters
    patience = cfg.EARLY_STOPPING_PATIENCE
    counter = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            # Backward + optimize
            loss.backward()
            optimizer.step()

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)

        print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc.item())

        # Validation phase
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(val_loader.dataset)
        epoch_acc = running_corrects.double() / len(val_loader.dataset)

        print(f'Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        history['val_loss'].append(epoch_loss)
        history['val_acc'].append(epoch_acc.item())

        # Update learning rate
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(epoch_acc)
            else:
                scheduler.step()
            history['lr'].append(optimizer.param_groups[0]['lr'])
            print(f'LR: {optimizer.param_groups[0]["lr"]:.7f}')

        # Check for overfitting
        if epoch > 3:
            train_acc = history['train_acc'][-1]
            val_acc = history['val_acc'][-1]

            if train_acc - val_acc > 0.2:
                print(f"\nWarning: Training accuracy ({train_acc:.4f}) much higher than validation accuracy ({val_acc:.4f})")
                print("Possible overfitting detected")

        # Save best model
        if epoch_acc > best_val_acc:
            best_val_acc = epoch_acc
            counter = 0
            torch.save(model.state_dict(), cfg.MODEL_SAVE_PATH)
            print(f"Saved best model with accuracy: {best_val_acc:.4f}")
        else:
            counter += 1
            print(f"EarlyStopping counter: {counter} out of {patience}")
            if counter >= patience:
                print("Early stopping")
                break

    # Plot training history
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title("Loss")
    plt.legend()

    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Val Accuracy')
    plt.title("Accuracy")
    plt.legend()

    # Plot overfitting analysis
    plt.subplot(2, 2, 3)
    train_val_gap = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
    plt.plot(train_val_gap, label='Train-Val Accuracy Gap')
    plt.axhline(y=0.1, color='r', linestyle='--', label='Mild Overfitting Threshold')
    plt.axhline(y=0.2, color='r', linestyle='-', label='Severe Overfitting Threshold')
    plt.title("Overfitting Analysis")
    plt.legend()

    # Plot learning rate if it's changing
    plt.subplot(2, 2, 4)
    if 'lr' in history:
        plt.plot(history['lr'], label='Learning Rate')
        plt.title("Learning Rate")
        plt.yscale('log')
    else:
        # If learning rate history not available, plot val_loss again
        plt.plot(history['val_loss'], label='Validation Loss')
        plt.title("Validation Loss")
    plt.legend()

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()

    return history

In [9]:
# 7. Evaluation Function
# --------------------

def evaluate_model(model, test_loader, class_to_idx=None):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics using scikit-learn
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print("\nFinal Test Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # Class-wise metrics if classes are provided
    if class_to_idx:
        idx_to_class = {v: k for k, v in class_to_idx.items()}
        class_correct = {}
        class_total = {}

        for i in range(len(all_preds)):
            label = all_labels[i]
            class_name = idx_to_class[label]

            if class_name not in class_total:
                class_total[class_name] = 0
                class_correct[class_name] = 0

            class_total[class_name] += 1
            if all_preds[i] == label:
                class_correct[class_name] += 1

        print("\nClass-wise Accuracy:")
        for class_name in class_total:
            acc = class_correct[class_name] / class_total[class_name] if class_total[class_name] > 0 else 0
            print(f"{class_name}: {acc:.4f} ({class_correct[class_name]}/{class_total[class_name]})")

    return acc, f1

In [10]:
# 8. Setting Random Seeds
# --------------------
def set_seeds(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed

In [11]:
# 9. Setting Seeds for Reproducibility
# ----------------------------------
seed = set_seeds(42)
print(f"Random seed set to {seed}")

Random seed set to 42


In [13]:
# import opendatasets as od
# od.download('https://www.kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset')

In [14]:
# 10. Set Dataset Paths
# ------------------
# IMPORTANT: Update these paths to match your file structure
base_path = "/content/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset"

# Paths for data
train_dir = os.path.join(base_path, "Training/training_words")
val_dir = os.path.join(base_path, "Validation/validation_words")
test_dir = os.path.join(base_path, "Testing/testing_words")

# Paths for labels
train_labels = os.path.join(base_path, "Training/training_labels.csv")
val_labels = os.path.join(base_path, "Validation/validation_labels.csv")
test_labels = os.path.join(base_path, "Testing/testing_labels.csv")

# Print paths to verify
print("Training directory:", train_dir)
print("Training labels:", train_labels)

Training directory: /content/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_words
Training labels: /content/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_labels.csv


In [ ]:
# 11. Initialize Datasets
# --------------------
# Training dataset
print("Initializing training dataset...")
train_dataset = PrescriptionDataset(
    train_dir,
    train_labels,
    img_size=cfg.IMG_SIZE,
    is_training=True
)

# Validation dataset
print("Initializing validation dataset...")
val_dataset = PrescriptionDataset(
    val_dir,
    val_labels,
    img_size=cfg.IMG_SIZE,
    is_training=False,
    class_to_idx=train_dataset.class_to_idx
)

# Test dataset
print("Initializing test dataset...")
test_dataset = PrescriptionDataset(
    test_dir,
    test_labels,
    img_size=cfg.IMG_SIZE,
    is_training=False,
    class_to_idx=train_dataset.class_to_idx
)

In [ ]:
# 12. Create Data Loaders
# --------------------
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Created data loaders with batch size {cfg.BATCH_SIZE}")

In [ ]:
# 13. Visualize Some Sample Images (Optional)
# ----------------------------------------
def visualize_samples(dataset, num_samples=5):
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))

    for i in range(num_samples):
        # Get a random sample
        idx = np.random.randint(0, len(dataset))
        img, label = dataset[idx]

        # Convert tensor to numpy for visualization
        img = img.numpy().transpose((1, 2, 0))

        # Denormalize
        img = img * np.array(cfg.STD) + np.array(cfg.MEAN)
        img = np.clip(img, 0, 1)

        # Plot
        axes[i].imshow(img)
        axes[i].set_title(f"Class: {dataset.idx_to_class[label]}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize training samples
visualize_samples(train_dataset)

In [ ]:
# 14. Create and Initialize Model
# ----------------------------
model = PrescriptionModel(len(train_dataset.class_to_idx)).to(device)
print(model)

In [ ]:
# 15. Define Loss Function and Optimizer
# -----------------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.3,
    patience=2,
    min_lr=1e-6,
    verbose=True
)

In [ ]:
# 16. Train the Model
# ----------------
print("Starting model training with overfitting prevention...")
history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    num_epochs=cfg.EPOCHS
)

In [ ]:
# 17. Load Best Model and Evaluate
# -----------------------------
# Load best model for evaluation
print("Loading best model for final evaluation...")
best_model = PrescriptionModel(len(train_dataset.class_to_idx)).to(device)
best_model.load_state_dict(torch.load(cfg.MODEL_SAVE_PATH))

# Evaluate on test set
print("Evaluating on test set...")
test_acc, test_f1 = evaluate_model(best_model, test_loader, train_dataset.class_to_idx)
print(f"Final results - Accuracy: {test_acc:.4f}, F1 Score: {test_f1:.4f}")

In [ ]:
# 18. Make Predictions on New Images (Example)
# -----------------------------------------
def predict_image(model, image_path, class_to_idx):
    # Create idx to class mapping
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    # Image preprocessing
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg.MEAN, std=cfg.STD)
    ])

    # Load and preprocess image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_tensor = transform(img)
    img_tensor = img_tensor.unsqueeze(0).to(device)  # Add batch dimension

    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        _, pred = torch.max(outputs, 1)
        predicted_class = idx_to_class[pred.item()]

        # Get probabilities
        probs = torch.nn.functional.softmax(outputs, dim=1)[0]
        top_probs, top_idxs = torch.topk(probs, 3)

    # Display results
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class}")
    plt.axis('off')
    plt.show()

    # Print top 3 predictions
    print("Top 3 predictions:")
    for i in range(3):
        print(f"{idx_to_class[top_idxs[i].item()]}: {top_probs[i].item():.4f}")

    return predicted_class

test_image_path = "/content/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Validation/validation_words/162.png"
prediction = predict_image(best_model, test_image_path, train_dataset.class_to_idx)